In [1]:
# Day 2 — GenAI-Assisted NetOps, Root-Cause Analysis & Infrastructure Automation

## Learning outcomes

By the end of the session, participants can:

1. apply GenAI to common network and infrastructure failure domains;
2. convert noisy operational events into a defensible incident narrative;
3. rank root-cause hypotheses using supporting and contradicting evidence;
4. run an evidence-driven AI incident war room;
5. generate infrastructure automation through a controlled lifecycle;
6. validate generated artifacts before testing or approval;
7. turn resolved incident knowledge into reusable operational assets.

In [ ]:
import ast
import io
import json
import os
import warnings
if not os.environ.get("LOKY_MAX_CPU_COUNT"):
    os.environ["LOKY_MAX_CPU_COUNT"] = "1"
warnings.filterwarnings("ignore", message=r"Could not find the number of physical cores.*", category=UserWarning)
import re
import sys
import time
from importlib.metadata import version
from pathlib import Path
from typing import Literal

import hcl2 # Used to read HashiCorp Configuration Language.
import pandas as pd
import yaml
from jinja2 import StrictUndefined, Template
from jsonschema import Draft202012Validator
from pydantic import BaseModel, Field
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

API_KEY_PRESENT = bool(os.getenv("OPENAI_API_KEY"))
LIVE_API = API_KEY_PRESENT and os.getenv("WAL_NET_ENABLE_LIVE_API", "1") == "1"
MODEL = os.getenv("WAL_NET_MODEL", "gpt-5.6-terra")
FAST_MODEL = os.getenv("WAL_NET_FAST_MODEL", "gpt-5.6-luna")
_client = None

def get_client():
    global _client
    if not LIVE_API:
        return None
    if _client is None:
        from openai import OpenAI
        _client = OpenAI()
    return _client

# call_text(
#     "RCA",
#     instructions="Analyze network evidence",
#     user_input="..."
# )
def call_text(
    name: str,
    *,
    instructions: str,
    user_input: str,
    model: str | None = None,
    reasoning_effort: str = "low",
    max_output_tokens: int = 700,
) -> dict | None:
    if not LIVE_API:
        print(f"[{name}] SKIPPED — provide OPENAI_API_KEY for a live response.")
        return None
    started = time.perf_counter()
    response = get_client().responses.create(
        model=model or FAST_MODEL,
        instructions=instructions,
        input=user_input,
        reasoning={"effort": reasoning_effort},
        max_output_tokens=max_output_tokens,
        store=False,
    )
    result = {
        "name": name,
        "model": response.model,
        "latency_seconds": round(time.perf_counter() - started, 3),
        "input_tokens": response.usage.input_tokens if response.usage else None,
        "output_tokens": response.usage.output_tokens if response.usage else None,
        "text": response.output_text,
    }
    print(f"\n--- {name} | {result['model']} | {result['latency_seconds']}s ---")
    print(result["text"])
    return result

def call_structured(
    name: str,
    schema: type[BaseModel],
    *,
    instructions: str,
    user_input: str,
    model: str | None = None,
    reasoning_effort: str = "low",
    show: bool = True,
):
    if not LIVE_API:
        print(f"[{name}] SKIPPED — live Structured Output requires OPENAI_API_KEY.")
        return None
    started = time.perf_counter()
    response = get_client().responses.parse(
        model=model or MODEL,
        instructions=instructions,
        input=user_input,
        reasoning={"effort": reasoning_effort},
        text_format=schema,
        store=False,
    )
    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError(f"{name}: no parsed output was returned")
    if show:
        print(f"\n--- {name} | {response.model} | {time.perf_counter() - started:.3f}s ---")
        print(parsed.model_dump_json(indent=2))
    return parsed

print("Python:", sys.version.split()[0])
print("Environment:", Path(sys.prefix).name)
print("openai:", version("openai"), "| pydantic:", version("pydantic"))
print("Live API:", LIVE_API, "| Model:", MODEL, "| Fast model:", FAST_MODEL)
if not API_KEY_PRESENT:
    print("NOTE: Live examples will show SKIPPED until OPENAI_API_KEY is configured.")

Python: 3.12.13
Environment: wal_net
openai: 3.3.1 | pydantic: 2.13.4
Live API: True | Model: gpt-5.6-terra | Fast model: gpt-5.6-luna


In [ ]:
# Imagine a company has a network problem:
#     “Users at one location suddenly cannot access an important application.”

# The immediate questions are:

# Is the network down?
# Is there a routing issue?
# Is DNS failing?
# Is the firewall blocking traffic?
# Was some configuration changed recently?
# Is there packet loss or latency?
# What evidence actually proves the likely cause?

# Normally, an engineer has to manually go through alerts, logs, telemetry, configuration files, recent changes, tickets, etc.

# This notebook demonstrates how GenAI can act like an engineering assistant to make this investigation faster.

# Problem Statement:
#     “How can we use GenAI to investigate a network incident, correlate large amounts of operational data, identify the most probable root cause, generate automation, and create operational documentation — while still keeping everything safe, validated and under human control?”

# Investigate → Correlate → Diagnose → Automate → Capture Knowledge

In [4]:
# 1. Investigate the incident

# Suppose an alert says:

# “Application connectivity has failed.”

# GenAI should not immediately say:

# “The router is the problem.”

# Instead, it considers different failure areas such as:

# Connectivity
# Routing/interface
# DNS/DHCP
# Latency/packet loss
# Firewall
# Configuration
# Availability

# The AI produces possible hypotheses and tells the engineer what additional evidence should be checked.

# So this stage answers:

# “Where should we start investigating?”

In [5]:
# 2. Correlate logs, alerts and events

# In production networks, the same incident may generate dozens or hundreds of alerts.

# For example:
# 09:41:58 → Application VIP unreachable
# 09:41:59 → Application VIP unreachable
# 09:42:04 → User reports application failure
# 09:42:12 → Routing log shows route missing
# 09:42:18 → WAN latency looks normal

# Instead of asking an engineer to manually understand everything, the notebook uses:

# duplicate removal,
# clustering,
# GenAI summarization,
# event correlation,
# timeline reconstruction.

# The goal is to convert:

# lots of noisy events

# into:

# a meaningful incident story.

In [6]:
# 3. Perform evidence-based Root Cause Analysis

# This is probably the core practical problem of the notebook.

# The notebook gives an incident:

# INC-2204

# and asks the AI to investigate it using:

# Alerts + Logs + Telemetry + Configuration + Recent Changes + Incident Context

# The AI has to generate multiple competing explanations, instead of jumping to one conclusion.

# For example:

# Hypothesis 1: WAN problem
# Hypothesis 2: Firewall problem
# Hypothesis 3: Routing/configuration problem

# Then evidence is used to eliminate or weaken hypotheses.

# For example, if:

# WAN latency is normal,
# packet loss is normal,
# firewall policy permits the traffic,
# a required route is missing,
# running configuration differs from the approved configuration,
# and a recent change removed a route-target,

# then the evidence increasingly points toward:

# a routing/configuration problem.

# So the notebook teaches:

# RCA should be based on evidence, not simply on what happened immediately before the incident.

In [7]:
# 4. Avoid the classic “correlation = causation” mistake

# Suppose somebody changed a router configuration at 9:40 AM and the application failed at 9:42 AM.

# It is tempting to say:

# “The configuration change caused the outage.”

# The notebook deliberately teaches that this is insufficient.

# You must establish a mechanism such as:

# Change removed route-target
# → required network prefix disappeared
# → application became unreachable.

# Only then does the change become strong causal evidence.

# Therefore:

# “A recent change happened before the incident” ≠ “The change caused the incident.”

In [8]:
# 5. Use GenAI to generate infrastructure automation safely

# Once engineers understand the problem, GenAI can help create things such as:

# Python scripts
# Shell scripts
# PowerShell
# Ansible
# Terraform
# YAML
# JSON
# REST API requests
# Configuration templates

# But the notebook treats AI-generated code as untrusted code.

In [9]:
# 6. Prevent AI from accidentally doing dangerous infrastructure actions

# Another important problem being addressed is:

# “What happens if AI generates dangerous automation?”

# The notebook explicitly checks for things such as:
# rm -rf
# reboot
# shutdown
# write memory
# configure terminal
# terraform apply

# and dangerous Python functions such as:
# eval()
# exec()
# system()
# popen()

# Architecture:
# GenAI
#   ↓
# Generates candidate
#   ↓
# Deterministic validation
#   ↓
# Safe testing
#   ↓
# Human approval
#   ↓
# Possible production use

# NOT
# GenAI → Production

In [10]:
# 7. Convert the investigation into reusable operational knowledge

# Finally, once the investigation is completed, GenAI helps create:

# Incident summary
# Change plan
# Troubleshooting document
# Runbook
# Post-incident/RCA report
# Configuration review
# Reusable knowledge article

# So instead of an engineer solving an issue once and that knowledge disappearing, the company can convert it into reusable operational documentation.

In [11]:
# Real Story:
#     A retail company's inventory application suddenly becomes unreachable at DC-07.

# Thousands of operational signals may exist.

# GenAI helps engineers:

# Step 1: Understand the symptoms.

# Step 2: Collect and summarize relevant alerts/logs.

# Step 3: Correlate events and create the incident timeline.

# Step 4: Generate different possible causes.

# Step 5: Compare those causes against actual network evidence.

# Step 6: Identify the most probable cause.

# Step 7: Generate safe diagnostic/automation scripts.

# Step 8: Validate and test those scripts before anyone can use them.

# Step 9: Generate RCA reports, runbooks and knowledge documents.

# But throughout the process:

# AI assists; deterministic systems validate; engineers remain accountable.

In [12]:
# Use GenAI as a NetOps copilot to investigate, correlate, diagnose and automate network incidents faster—but never trust AI output blindly; every important claim and automation must be backed by evidence, deterministic validation, testing and human approval.

# Module 1 — AI-Assisted Network & Infrastructure Troubleshooting

## 1.1 Failure domains and the role of GenAI

| Failure domain | Typical symptoms | Evidence an engineer needs | Useful GenAI contribution | Boundary |
|---|---|---|---|---|
| Connectivity failures | Timeout, unreachable service, failed session | Source/destination, path, interface state, reachability tests | Normalize symptoms and propose layer-by-layer checks | Never equate “timeout” with one cause |
| Routing and interface issues | Missing prefix, adjacency reset, interface errors | RIB/FIB, neighbor state, counters, optics, topology | Relate interface events to routing impact and rank tests | Timing alone does not prove causality |
| DNS/DHCP problems | Lookup timeout, wrong answer, lease failure | Resolver/relay health, scopes, ACLs, client configuration | Separate naming/addressing failures from application symptoms | Do not invent server health or lease state |
| Latency and packet loss | Slow transaction, jitter, retransmission | Baselines, hop/path measurements, queue and interface telemetry | Compare scope and time windows; summarize anomalies | Averages can hide tail latency and burst loss |
| Firewall/connectivity problems | Denied flow, reset, asymmetric connection | Policy, hit counters, flow logs, NAT state, approved intent | Explain rules, connect counters to affected flows | A matching rule is not automatically the cause |
| Configuration inconsistencies | Site-specific drift, missing parameter, unexpected behavior | Intended state, running state, version, recent change | Compare configurations and explain the material difference | Generated syntax must be independently validated |
| Infrastructure availability issues | Service or device unavailable, dependency cascade | Health checks, dependency graph, redundancy state, maintenance | Summarize blast radius and competing dependency failures | Do not claim recovery without post-action evidence |

### AI-assisted troubleshooting workflow

**Scope incident → collect trusted evidence → normalize → generate hypotheses → identify contradictions → choose read-only checks → update confidence → recommend → human validation**

GenAI is most valuable in the language-and-reasoning layer. Authoritative operational tools remain the source of current state.

In [13]:
class EvidenceClaim(BaseModel):
    statement: str
    evidence_ids: list[str] = Field(min_length=1)

class TriageHypothesis(BaseModel):
    cause: str
    confidence: Literal["low", "medium", "high"]
    supporting_evidence: list[str]
    missing_evidence: list[str]
    next_read_only_check: str

class TriageCase(BaseModel):
    incident_id: str
    primary_domain: Literal[
        "connectivity", "routing_interface", "dns_dhcp", "latency_loss",
        "firewall", "configuration", "availability"
    ]
    symptoms: list[EvidenceClaim]
    hypotheses: list[TriageHypothesis]
    immediate_escalation: bool

class TriageBatch(BaseModel):
    cases: list[TriageCase]

FAILURE_DOMAIN_CASES = """
CASE INC-101 | E1: Checkout clients in SITE-101 time out to payment-gateway.internal; E2: gateway health is not supplied.
CASE INC-102 | E1: DC-02 edge interface xe-0/0/7 flapped; E2: BGP neighbor reset eight seconds later; E3: optic peer data missing.
CASE INC-103 | E1: SITE-207 clients receive no DHCP lease; E2: relay counters increase; E3: scope utilization unavailable.
CASE INC-104 | E1: Voice-picking p95 latency rose from 80 ms to 610 ms; E2: average WAN loss is 0.2%; E3: queue telemetry missing.
CASE INC-105 | E1: Firewall deny counter increases for client-to-resolver UDP/53; E2: rule intent not supplied.
CASE INC-106 | E1: RETAIL VRF at DC-07 lacks prefix 10.90.40.0/24; E2: route-target differs from intended template.
CASE INC-107 | E1: Inventory API is unhealthy in one region; E2: database and network dependency health are not supplied.
""".strip()

In [15]:
FAILURE_DOMAIN_CASES

'CASE INC-101 | E1: Checkout clients in SITE-101 time out to payment-gateway.internal; E2: gateway health is not supplied.\nCASE INC-102 | E1: DC-02 edge interface xe-0/0/7 flapped; E2: BGP neighbor reset eight seconds later; E3: optic peer data missing.\nCASE INC-103 | E1: SITE-207 clients receive no DHCP lease; E2: relay counters increase; E3: scope utilization unavailable.\nCASE INC-104 | E1: Voice-picking p95 latency rose from 80 ms to 610 ms; E2: average WAN loss is 0.2%; E3: queue telemetry missing.\nCASE INC-105 | E1: Firewall deny counter increases for client-to-resolver UDP/53; E2: rule intent not supplied.\nCASE INC-106 | E1: RETAIL VRF at DC-07 lacks prefix 10.90.40.0/24; E2: route-target differs from intended template.\nCASE INC-107 | E1: Inventory API is unhealthy in one region; E2: database and network dependency health are not supplied.'

In [14]:
EXPECTED_DOMAIN_BY_INCIDENT = {
    "INC-101": "connectivity",
    "INC-102": "routing_interface",
    "INC-103": "dns_dhcp",
    "INC-104": "latency_loss",
    "INC-105": "firewall",
    "INC-106": "configuration",
    "INC-107": "availability",
}

ALLOWED_TRIAGE_EVIDENCE = {
    "INC-101": {"E1", "E2"},
    "INC-102": {"E1", "E2", "E3"},
    "INC-103": {"E1", "E2", "E3"},
    "INC-104": {"E1", "E2", "E3"},
    "INC-105": {"E1", "E2"},
    "INC-106": {"E1", "E2"},
    "INC-107": {"E1", "E2"},
}

def normalize_known_training_domains(batch: TriageBatch) -> tuple[TriageBatch, list[str]]:
    """Apply the exercise's declared taxonomy; record rather than hide model corrections."""
    corrections = []
    normalized = []
    for case in batch.cases:
        expected = EXPECTED_DOMAIN_BY_INCIDENT.get(case.incident_id)
        if expected and case.primary_domain != expected:
            corrections.append(f"{case.incident_id}: {case.primary_domain} -> {expected}")
            case = case.model_copy(update={"primary_domain": expected})
        normalized.append(case)
    return batch.model_copy(update={"cases": normalized}), corrections

def validate_triage_batch(batch: TriageBatch) -> list[str]:
    problems = []
    expected_ids = set(EXPECTED_DOMAIN_BY_INCIDENT)
    actual_ids = [case.incident_id for case in batch.cases]
    if set(actual_ids) != expected_ids or len(actual_ids) != len(expected_ids):
        problems.append(f"Expected each canonical incident exactly once; received {actual_ids}")

    for case in batch.cases:
        expected_domain = EXPECTED_DOMAIN_BY_INCIDENT.get(case.incident_id)
        if expected_domain is None:
            problems.append(f"Unknown incident ID: {case.incident_id}")
            continue
        if case.primary_domain != expected_domain:
            problems.append(
                f"{case.incident_id} must map to {expected_domain}, not {case.primary_domain}"
            )
        if not case.hypotheses:
            problems.append(f"{case.incident_id} has no hypothesis")
        cited = set()
        for symptom in case.symptoms:
            cited.update(symptom.evidence_ids)
        for hypothesis in case.hypotheses:
            cited.update(hypothesis.supporting_evidence)
            if not hypothesis.next_read_only_check.strip():
                problems.append(f"{case.incident_id} has no next read-only check")
        unknown = cited - ALLOWED_TRIAGE_EVIDENCE[case.incident_id]
        if unknown:
            problems.append(f"{case.incident_id} cites unsupported evidence: {sorted(unknown)}")
    return sorted(set(problems))

triage_batch = call_structured(
    "Module 1 — seven-domain AI-assisted triage",
    TriageBatch,
    model=FAST_MODEL,
    instructions=(
        "Triage every case using only its evidence IDs. The canonical training taxonomy is: "
        "INC-101=connectivity, INC-102=routing_interface, INC-103=dns_dhcp, "
        "INC-104=latency_loss, INC-105=firewall, INC-106=configuration, "
        "INC-107=availability. Return every incident exactly once. Do not invent command results, "
        "configuration, server health, or confirmed causes. Give concise hypotheses and the smallest "
        "useful read-only check."
    ),
    user_input=FAILURE_DOMAIN_CASES,
    show=False,
)

accepted_triage_batch = None
triage_problems = ["Live triage was not generated."]
if triage_batch:
    triage_batch, domain_corrections = normalize_known_training_domains(triage_batch)
    if domain_corrections:
        print("CONTROLLED TAXONOMY CORRECTIONS:")
        for correction in domain_corrections:
            print("-", correction)
    triage_problems = validate_triage_batch(triage_batch)
    print("TRIAGE RELEASE:", "PASS" if not triage_problems else "REJECT")
    for problem in triage_problems:
        print("-", problem)
    if not triage_problems:
        accepted_triage_batch = triage_batch
        print(accepted_triage_batch.model_dump_json(indent=2))
        triage_rows = []
        for case in accepted_triage_batch.cases:
            triage_rows.append({
                "incident_id": case.incident_id,
                "domain": case.primary_domain,
                "top_hypothesis": case.hypotheses[0].cause,
                "confidence": case.hypotheses[0].confidence,
                "next_check": case.hypotheses[0].next_read_only_check,
            })
        display(pd.DataFrame(triage_rows))
else:
    print("TRIAGE RELEASE: SKIPPED — no live response is available.")

TRIAGE RELEASE: PASS
{
  "cases": [
    {
      "incident_id": "INC-101",
      "primary_domain": "connectivity",
      "symptoms": [
        {
          "statement": "Checkout clients in SITE-101 time out to payment-gateway.internal.",
          "evidence_ids": [
            "E1"
          ]
        },
        {
          "statement": "Gateway health information is unavailable.",
          "evidence_ids": [
            "E2"
          ]
        }
      ],
      "hypotheses": [
        {
          "cause": "Connectivity failure between SITE-101 and the payment gateway, with gateway-side reachability still unverified.",
          "confidence": "medium",
          "supporting_evidence": [
            "E1",
            "E2"
          ],
          "missing_evidence": [
            "E2"
          ],
          "next_read_only_check": "Review path and reachability telemetry from SITE-101 to payment-gateway.internal."
        }
      ],
      "immediate_escalation": true
    },
    {
      "inc

,incident_id,domain,top_hypothesis,confidence,next_check
0,INC-101,connectivity,Connectivity failure between SITE-101 and the ...,medium,Review path and reachability telemetry from SI...
1,INC-102,routing_interface,The interface flap may have caused the subsequ...,high,"Review interface flap, optic, and BGP event hi..."
2,INC-103,dns_dhcp,The relay is forwarding or observing DHCP acti...,medium,Review DHCP server allocation and scope-availa...
3,INC-104,latency_loss,Severe latency degradation is present; queuein...,medium,"Review hop-level latency, loss, and queue tele..."
4,INC-105,firewall,Firewall policy is denying client-to-resolver ...,high,"Inspect the matching firewall rule, hit detail..."
5,INC-106,configuration,A route-target configuration mismatch may prev...,high,Compare the DC-07 RETAIL VRF route-targets and...
6,INC-107,availability,Regional Inventory API availability failure wi...,high,"Review regional API, database, and network dep..."


# Module 2 — Intelligent Log, Alert & Event Analysis

## 2.1 From event volume to engineering meaning

| Capability | What it does | Engineering value | Common failure mode |
|---|---|---|---|
| Log summarization | Compresses repeated messages while preserving material facts | Faster situational awareness | Drops timestamps, scope or negative evidence |
| Alert enrichment | Adds service, site, severity, owner and evidence context | Makes an alert actionable | Enrichment source is stale or unauthorized |
| Event correlation | Groups events by time, topology, service and shared change | Reveals a candidate incident chain | Treats temporal proximity as causation |
| Alert-noise reduction | Deduplicates and suppresses known non-actionable repeats | Reduces cognitive load | Hides a symptom that changed severity or scope |
| Incident clustering | Groups semantically or operationally related events | Finds incident candidates across sources | Similar wording creates a false cluster |
| Timeline reconstruction | Orders evidence and marks clock/source gaps | Supports causal reasoning and handoffs | Mixed time zones or delayed ingestion distort order |
| Engineering narrative | Converts events into a concise, evidence-linked account | Improves war-room alignment | Produces a persuasive story with unsupported links |

Use deterministic filtering for exact duplicate IDs, maintenance windows and explicit suppression rules. Use GenAI after this first reduction to summarize, enrich and reason over the retained evidence.

In [17]:
EVENTS = [
    {"id":"EV1","time":"09:41:58","site":"DC-07","source":"alert","severity":"critical","text":"Inventory VIP reachability from RETAIL VRF dropped to 0%."},
    {"id":"EV2","time":"09:41:59","site":"DC-07","source":"alert","severity":"critical","text":"Inventory VIP reachability from RETAIL VRF dropped to 0%."},
    {"id":"EV3","time":"09:42:04","site":"DC-07","source":"ticket","severity":"high","text":"Picking handhelds authenticate but cannot open inventory application."},
    {"id":"EV4","time":"09:42:12","site":"DC-07","source":"routing","severity":"high","text":"Prefix 10.90.40.0/24 absent from RETAIL VRF; BGP sessions established."},
    {"id":"EV5","time":"09:42:18","site":"DC-07","source":"telemetry","severity":"info","text":"WAN RTT 22 ms and packet loss 0.1%, within baseline."},
    {"id":"EV6","time":"09:42:26","site":"DC-07","source":"change","severity":"warning","text":"CHG-204 completed at 09:36; VRF import policy changed."},
    {"id":"EV7","time":"09:43:00","site":"SITE-114","source":"alert","severity":"warning","text":"Guest Wi-Fi client count exceeded forecast; service healthy."},
    {"id":"EV8","time":"09:43:05","site":"DC-07","source":"firewall","severity":"info","text":"Policy simulation permits handheld subnet to inventory VIP."},
]

pd.set_option("display.max_colwidth", None)
events_df = pd.DataFrame(EVENTS).sort_values("time")
display(events_df)

,id,time,site,source,severity,text
0,EV1,09:41:58,DC-07,alert,critical,Inventory VIP reachability from RETAIL VRF dropped to 0%.
1,EV2,09:41:59,DC-07,alert,critical,Inventory VIP reachability from RETAIL VRF dropped to 0%.
2,EV3,09:42:04,DC-07,ticket,high,Picking handhelds authenticate but cannot open inventory application.
3,EV4,09:42:12,DC-07,routing,high,Prefix 10.90.40.0/24 absent from RETAIL VRF; BGP sessions established.
4,EV5,09:42:18,DC-07,telemetry,info,"WAN RTT 22 ms and packet loss 0.1%, within baseline."
5,EV6,09:42:26,DC-07,change,warning,CHG-204 completed at 09:36; VRF import policy changed.
6,EV7,09:43:00,SITE-114,alert,warning,Guest Wi-Fi client count exceeded forecast; service healthy.
7,EV8,09:43:05,DC-07,firewall,info,Policy simulation permits handheld subnet to inventory VIP.


In [18]:
# Exact duplicate reduction is deterministic; semantic clustering is an analytical aid.
deduped = events_df.drop_duplicates(subset=["site", "source", "text"], keep="first").copy()

vectorizer = TfidfVectorizer(stop_words="english")
event_vectors = vectorizer.fit_transform(deduped["text"])
cluster_count = min(3, len(deduped))
deduped["cluster"] = KMeans(n_clusters=cluster_count, random_state=42, n_init=10).fit_predict(event_vectors)

print(f"Raw events: {len(events_df)} | After exact deduplication: {len(deduped)}")
display(deduped[["id", "time", "site", "source", "severity", "cluster", "text"]])

Raw events: 8 | After exact deduplication: 7


,id,time,site,source,severity,cluster,text
0,EV1,09:41:58,DC-07,alert,critical,0,Inventory VIP reachability from RETAIL VRF dropped to 0%.
2,EV3,09:42:04,DC-07,ticket,high,0,Picking handhelds authenticate but cannot open inventory application.
3,EV4,09:42:12,DC-07,routing,high,0,Prefix 10.90.40.0/24 absent from RETAIL VRF; BGP sessions established.
4,EV5,09:42:18,DC-07,telemetry,info,1,"WAN RTT 22 ms and packet loss 0.1%, within baseline."
5,EV6,09:42:26,DC-07,change,warning,0,CHG-204 completed at 09:36; VRF import policy changed.
6,EV7,09:43:00,SITE-114,alert,warning,2,Guest Wi-Fi client count exceeded forecast; service healthy.
7,EV8,09:43:05,DC-07,firewall,info,0,Policy simulation permits handheld subnet to inventory VIP.


In [19]:
class EnrichedAlert(BaseModel):
    event_id: str
    normalized_symptom: str
    affected_scope: str
    related_evidence_ids: list[str]
    missing_context: list[str]

class CorrelationGroup(BaseModel):
    group_name: str
    evidence_ids: list[str]
    relationship: str
    causal_status: Literal["not_assessed", "correlated", "causally_supported"]

class NarrativeEvent(BaseModel):
    time: str
    description: str
    evidence_ids: list[str]

class EventIntelligenceReport(BaseModel):
    summary: str
    enriched_alerts: list[EnrichedAlert]
    noise_candidates: list[str]
    correlation_groups: list[CorrelationGroup]
    incident_clusters: list[list[str]]
    timeline: list[NarrativeEvent]
    engineering_narrative: str
    unknowns: list[str]

In [20]:
event_context = "\n".join(
    f"{row.id} | {row.time} | {row.site} | {row.source} | {row.severity} | {row.text}"
    for row in deduped.itertuples(index=False)
)

event_intelligence = call_structured(
    "Module 2 — log, alert and event intelligence",
    EventIntelligenceReport,
    instructions=(
        "Transform retained operational events into an evidence-linked engineering account. "
        "Use only supplied event IDs. Keep unrelated sites separate. Correlation is not causation. "
        "Treat normal telemetry and permits as potentially contradicting evidence, not noise."
    ),
    user_input=event_context,
)


--- Module 2 — log, alert and event intelligence | gpt-5.6-terra | 17.349s ---
{
  "summary": "DC-07 experienced an inventory-application reachability incident affecting the RETAIL VRF: the inventory VIP became unreachable and picking handhelds could authenticate but could not open the application. The strongest evidence is the simultaneous absence of prefix 10.90.40.0/24 from the RETAIL VRF following a VRF import-policy change. WAN health and firewall-policy simulation provide contradictory evidence against WAN impairment or an explicit handheld-to-VIP firewall denial. SITE-114 is a separate, healthy guest Wi-Fi capacity event.",
  "enriched_alerts": [
    {
      "event_id": "EV1",
      "normalized_symptom": "Inventory VIP unreachable from the RETAIL VRF",
      "affected_scope": "DC-07 RETAIL VRF clients accessing the inventory VIP",
      "related_evidence_ids": [
        "EV3",
        "EV4",
        "EV5",
        "EV6",
        "EV8"
      ],
      "missing_context": [
       

In [21]:
def validate_event_intelligence(report: EventIntelligenceReport, allowed_ids: set[str]) -> list[str]:
    problems = []
    cited = set()
    for alert in report.enriched_alerts:
        cited.add(alert.event_id)
        cited.update(alert.related_evidence_ids)
    for group in report.correlation_groups:
        cited.update(group.evidence_ids)
        if group.causal_status == "causally_supported" and len(set(group.evidence_ids)) < 2:
            problems.append(f"Causal group lacks converging evidence: {group.group_name}")
    for event in report.timeline:
        cited.update(event.evidence_ids)
    for cluster in report.incident_clusters:
        cited.update(cluster)
    unknown = cited - allowed_ids
    if unknown:
        problems.append(f"Unknown event IDs: {sorted(unknown)}")
    unrelated_noise = [eid for eid in report.noise_candidates if eid not in allowed_ids]
    if unrelated_noise:
        problems.append(f"Noise list contains unknown IDs: {unrelated_noise}")
    return problems

if event_intelligence:
    event_problems = validate_event_intelligence(event_intelligence, set(deduped["id"]))
    print("EVENT REPORT:", "PASS" if not event_problems else "REJECT")
    for problem in event_problems:
        print("-", problem)

EVENT REPORT: PASS


# Module 3 — Evidence-Driven Root-Cause Analysis

## 3.1 RCA is a falsifiable argument, not a story

| Topic | Practical meaning |
|---|---|
| Multi-source evidence collection | Combine alerts, logs, telemetry, configuration, change history and incident context with source and timestamp preserved |
| Hypothesis generation | Produce plausible alternatives that explain the observed scope—not merely variants of one favored answer |
| Hypothesis ranking | Rank by explanatory coverage, independent support, contradictions and missing discriminating evidence |
| Correlation vs causation | A change before impact is correlated; causation requires a mechanism and evidence that the mechanism affected the observed service |
| Supporting and contradicting evidence | Record both. Healthy WAN telemetry or a permitting firewall policy can actively weaken competing hypotheses |
| Confidence and uncertainty | Confidence should fall when key evidence is missing, stale, contradictory or from one source |
| Preventing hallucinated RCA | Constrain evidence IDs, schema, causal status and execution authority; reject unsupported precision |
| Human validation | An accountable engineer reviews source quality, operational applicability, blast radius and proposed next action |

### Confidence guide

| Confidence | Evidence expectation | Appropriate language |
|---|---|---|
| Low | Symptom fit with major gaps or contradictions | “Possible; collect…” |
| Medium | Multiple supporting items but mechanism or scope incomplete | “Probable; validate…” |
| High | Converging independent sources, explicit mechanism and limited contradiction | “Strongly supported…” |

Even “high” model confidence does not equal a confirmed root cause or permission to remediate.